# Random forest regression in OMLE

Train, evaluate, export, inspect, validate, visualize, and verify a complete fitted model. Run all cells from a fresh kernel.

## Dataset

**[Diabetes](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html)** — Predict a quantitative measure of disease progression one year after baseline. The loader provides 442 samples and 10 features; its default feature normalization is part of this dataset definition. This is an educational example, not a clinically validated model.

Data is loaded directly through scikit-learn; no dataset directory is required. The loader description below preserves additional source details and attribution.

In [ ]:
import json
import platform
from pathlib import Path
from importlib.metadata import version

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

import omle
import omle_runtime as omr
from omle_convert import export_omle

SEED = 42
ATOL = 1e-4
RTOL = 1e-4
packages = ['numpy', 'pandas', 'scikit-learn', 'omle', 'omle-convert', 'omle-runtime', 'omle-viewer']
environment = {"python": platform.python_version(), "packages": {p: version(p) for p in packages}}
display(environment)

## Load and split

Reserve a fixed 20% test split. Fit the model and any preprocessing on training rows only. No hyperparameter tuning uses the held-out test set.

Use float32 features throughout to match scikit-learn tree inference precision, including threshold-boundary behavior. The OMLE input schema records this dtype.

In [ ]:
dataset = load_diabetes(as_frame=True)
# sklearn tree prediction uses float32 features internally. Declare the
# same float32 interface for native inference and the exported artifact.
X, y = dataset.data.astype(np.float32), dataset.target
print(dataset.DESCR)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)
print(f"Features: {X.shape[1]}; training rows: {len(X_train)}; test rows: {len(X_test)}")
display(X_train.head())

## Train and evaluate

Evaluate quality on the held-out test set. These quality metrics are separate from the OMLE conversion fidelity checked later.

In [ ]:
pipeline = Pipeline([("regressor", RandomForestRegressor(n_estimators=50, max_depth=6, random_state=SEED, n_jobs=1))])
pipeline.fit(X_train, y_train)
native_labels = pipeline.predict(X_test)

In [ ]:
metrics = {
    "mae": float(mean_absolute_error(y_test, native_labels)),
    "rmse": float(np.sqrt(mean_squared_error(y_test, native_labels))),
    "r2": float(r2_score(y_test, native_labels)),
}
display(pd.Series(metrics, name="held_out_metrics"))

## Export the fitted pipeline

Export the entire fitted pipeline, including preprocessing when present. Embedded verification examples use training rows; the independent parity check uses every held-out row. Rerunning overwrites this recipe’s own generated artifacts.

In [ ]:
cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents]
                  if (p / "requirements.txt").is_file() and (p / "notebooks").is_dir()), None)
if repo_root is None:
    raise RuntimeError("Start Jupyter from omle-models or a notebook subdirectory.")
output_dir = repo_root / "models" / "regression" / "diabetes"
output_dir.mkdir(parents=True, exist_ok=True)
model_path = output_dir / "sklearn-random-forest.omle"
export_omle(
    pipeline, str(model_path), X=X_train,
    model_name="diabetes-sklearn-random-forest", model_version="0.1.0",
    random_state=SEED, n_verify=10, n_sample=3, verify_atol=ATOL, verify_rtol=RTOL,
)
print(f"Exported {model_path.relative_to(repo_root)} ({model_path.stat().st_size:,} bytes)")

## Inspect the exported model

Inspect declared inputs, outputs, and operators before running structural validation. A `-1` shape dimension denotes a variable batch size.

In [ ]:
import omle

exported_model = omle.load(str(model_path))

print("Model inputs")
display(pd.DataFrame([
    {
        "name": spec.name,
        "dtype": spec.type.dtype.name if spec.type else None,
        "shape": spec.type.shape if spec.type else None,
    }
    for spec in exported_model.inputs
]))

print("Model outputs")
display(pd.DataFrame([
    {
        "name": spec.name,
        "dtype": spec.type.dtype.name if spec.type else None,
        "shape": spec.type.shape if spec.type else None,
        "role": spec.role.name,
    }
    for spec in exported_model.outputs
]))

### Operators included in the model

Show the actual graph operators and their imported namespace versions.

In [ ]:
operator_versions = {
    item.namespace: item.version for item in exported_model.operator_imports
}
operators = pd.DataFrame([
    {
        "node": node.name,
        "namespace": node.domain,
        "operator": node.op,
        "version": operator_versions.get(node.domain),
    }
    for node in exported_model.nodes
])
display(operators)

### Validate the model

Structural validation checks graph and operator declarations. Errors stop execution; warnings remain visible. Numeric prediction parity is checked separately below.

In [ ]:
validation = omle.validate(exported_model)
print(validation)
if not validation.is_valid:
    raise ValueError(f"Exported OMLE model is invalid:\n{validation}")

### Explore the interactive graph

Select the graph nodes to inspect their fitted parameters and connections. The viewer is embedded locally; no model upload or external viewer server is needed.

Run the cell below in JupyterLab or your IDE notebook editor to interact with the graph. GitHub’s static notebook preview will not run it.

In [ ]:
from omle_viewer import show

viewer = show(exported_model, height=650)

## Verify OMLE inference

Run the artifact on raw test features in their original column order. Compare with this model’s native implementation, not a separately trained model from another framework.

In [ ]:
runtime_model = omr.load(str(model_path))
omle_labels = runtime_model.predict(X_test)
assert omle_labels.shape == native_labels.shape
assert np.isfinite(omle_labels).all()
np.testing.assert_allclose(omle_labels, native_labels, atol=ATOL, rtol=RTOL)
parity = {"test_rows": len(X_test), "max_absolute_prediction_error": float(np.max(np.abs(omle_labels - native_labels)))}
parity.update({"atol": ATOL, "rtol": RTOL,
               "native_prediction_dtype": str(native_labels.dtype),
               "omle_prediction_dtype": str(omle_labels.dtype)})
display(pd.Series(parity, name="conversion_fidelity"))
print("PASS: OMLE predictions match the native fitted model.")

## Save the run record

Save exact dependency versions, dataset identity, feature order, training configuration, split membership, quality metrics, and conversion fidelity next to this model. Generated artifacts remain ignored by Git. A fixed seed alone does not guarantee identical fits across dependency versions.

In [ ]:
report = {
    "environment": environment,
    "dataset": "sklearn.datasets.load_diabetes", "dataset_source": "https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html",
    "seed": SEED, "test_size": 0.2,
    "feature_names": list(X.columns), "input_dtype": str(X.dtypes.iloc[0]),
    "train_indices": X_train.index.tolist(), "test_indices": X_test.index.tolist(),
    "model_parameters": {str(k): str(v) for k, v in pipeline.get_params(deep=True).items()},
    "metrics": metrics, "parity": parity,
}
report_path = output_dir / "sklearn-random-forest-environment.json"
report_path.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print(f"Saved {report_path.relative_to(repo_root)}")